# Short Notebook 2

Group: 10 \
Kaggle name: [10] Vintage ML in Theory \
Team Members: Henrik Remmen (541747), Elias Melkeraaen (563433) and Christian Økland (560367) 


This code implements a global LightGBM-based model using the Darts framework, designed to forecast daily material receivals for each `rm_id`.


## 1 - Imports

In [11]:
import warnings
import numpy as np
import pandas as pd
import os
from pathlib import Path
from typing import List

from darts import TimeSeries
from darts.models import XGBModel

# --- demp støy ---
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names", category=UserWarning)
os.environ["LIGHTGBM_VERBOSE"] = "0"
os.environ["LIGHTGBM_VERBOSITY"] = "0"

## 2 - Files and Constants 

In [12]:
# --- Paths ---
DATA_DIR = Path('./data')
PREDICTION_MAPPING_PATH = DATA_DIR / 'prediction_mapping.csv'
SAMPLE_SUB_PATH = DATA_DIR / 'sample_submission.csv'
RECEIVALS_PATH = Path("./data/kernel/receivals.csv")
PURCHASE_ORDERS_PATH = Path("./data/kernel/purchase_orders.csv")
MATERIALS_PATH = Path("./data/extended/materials.csv")

FORECAST_START = pd.Timestamp("2025-01-01")
FORECAST_END   = pd.Timestamp("2025-05-31")
TRAIN_END      = pd.Timestamp("2024-12-19")
FEATURE_END    = TRAIN_END
RECEIVALS_MIN_DATE   = pd.Timestamp("2010-01-01")
RECEIVALS_MAX_CUTOFF = None

INCLUDE_RM_IDS = {
    2130, 2131, 2132, 2134, 2135, 2142, 2143, 2144, 2145,
    2741, 3122, 3123, 3124, 3125, 3126, 
    3282, 3362, 3421, 3781, 3865, 3901, 4222
}


## 3 - Data preprocessing 

In [13]:
def _to_datetime_col(df: pd.DataFrame, col: str, dayfirst: bool | None = None) -> pd.Series:
    if col not in df.columns:
        raise KeyError(f"Kolonnen '{col}' finnes ikke i DataFrame.")
    s = df[col]
    if np.issubdtype(s.dtype, np.number):
        dt = pd.to_datetime(s, unit="ms", errors="coerce")
        if dt.isna().all():
            dt = pd.to_datetime(s, unit="s", errors="coerce")
    else:
        dt = pd.to_datetime(s, errors="coerce", dayfirst=dayfirst, infer_datetime_format=True, utc=True)
    if hasattr(dt, "dt"):
        dt = dt.dt.tz_localize(None).dt.normalize()
    return dt

def load_receivals() -> pd.DataFrame:
    df = pd.read_csv(RECEIVALS_PATH).copy()
    df.columns = df.columns.str.strip()
    if "date_arrival" in df.columns:
        df["actual_delivery_date"] = _to_datetime_col(df, "date_arrival")
    else:
        df["actual_delivery_date"] = _to_datetime_col(df, "actual_delivery_date")
    keep = ["rm_id", "net_weight", "actual_delivery_date", "purchase_order_id", "purchase_order_item_no"]
    df = df[[c for c in keep if c in df.columns]].copy()
    for c in ["rm_id","net_weight","purchase_order_id","purchase_order_item_no"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.rename(columns={"actual_delivery_date": "date_arrival"})
    df["date_arrival"] = _to_datetime_col(df, "date_arrival")
    if "net_weight" not in df.columns:
        df["net_weight"] = 0.0
    return df


def load_purchase_orders() -> pd.DataFrame:
    if not PURCHASE_ORDERS_PATH.exists():
        return pd.DataFrame(columns=["purchase_order_id","purchase_order_item_no","delivery_date"])
    po = pd.read_csv(PURCHASE_ORDERS_PATH).copy()
    po.columns = po.columns.str.strip()
    keep = ["purchase_order_id","purchase_order_item_no","delivery_date"]
    po = po[[c for c in keep if c in po.columns]].copy()
    if "delivery_date" in po.columns:
        po["delivery_date"] = _to_datetime_col(po, "delivery_date")
    for c in ["purchase_order_id","purchase_order_item_no"]:
        if c in po.columns:
            po[c] = pd.to_numeric(po[c], errors="coerce")
    return po


def load_materials_latest() -> pd.DataFrame:
    candidate_paths = [Path("./data/materials.csv"), MATERIALS_PATH]
    mat_path = next((p for p in candidate_paths if Path(p).exists()), None)
    if mat_path is None:
        return pd.DataFrame(columns=["rm_id","raw_material_alloy","raw_material_format_type","stock_location"])

    df = pd.read_csv(mat_path).copy()
    df.columns = df.columns.str.strip()

    if "rm_id" not in df.columns:
        return pd.DataFrame(columns=["rm_id","raw_material_alloy","raw_material_format_type","stock_location"])

    df["rm_id"] = pd.to_numeric(df["rm_id"], errors="coerce")
    df = df[df["rm_id"].notna()].copy()
    df["rm_id"] = df["rm_id"].astype(int)

    for col in ["raw_material_alloy", "raw_material_format_type", "stock_location"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().replace("", "UNKNOWN").fillna("UNKNOWN")
        else:
            df[col] = "UNKNOWN"

    if "product_version" in df.columns:
        df["product_version"] = pd.to_numeric(df["product_version"], errors="coerce").fillna(-1)
        df_latest = (
            df.sort_values(["rm_id", "product_version"], kind="mergesort")
              .drop_duplicates(subset=["rm_id"], keep="last")
        )
    else:
        df_latest = df.drop_duplicates(subset=["rm_id"], keep="last")

    return df_latest[["rm_id","raw_material_alloy","raw_material_format_type","stock_location"]]


def encode_three_material_columns_compact(materials_df: pd.DataFrame) -> pd.DataFrame:
    if materials_df.empty:
        return pd.DataFrame(columns=["rm_id","alloy_code","format_type_code","location_code"])
    df = materials_df.copy()
    for col in ["raw_material_alloy","raw_material_format_type","stock_location"]:
        if col not in df.columns:
            df[col] = "UNKNOWN"
        df[col] = df[col].astype(str).str.strip().replace("", "UNKNOWN").fillna("UNKNOWN")

    def compact_factorize(s: pd.Series) -> pd.Series:
        unique_vals = [v for v in s.unique() if v != "UNKNOWN"]
        mapping = {v: i + 1 for i, v in enumerate(sorted(unique_vals))}
        mapping["UNKNOWN"] = 0
        return s.map(mapping).fillna(0).astype(int)

    df["alloy_code"]        = compact_factorize(df["raw_material_alloy"])
    df["format_type_code"]  = compact_factorize(df["raw_material_format_type"])
    df["location_code"]     = compact_factorize(df["stock_location"])
    return df[["rm_id","alloy_code","format_type_code","location_code"]]


def build_daily_material_signals_fast(receivals_df: pd.DataFrame, materials_enc_df: pd.DataFrame) -> pd.DataFrame:
    if receivals_df.empty or materials_enc_df.empty:
        return pd.DataFrame(columns=["rm_id","date","alloy_code","format_type_code","location_code"])
    rec = receivals_df.copy()
    rec["date"] = pd.to_datetime(rec["date_arrival"]).dt.normalize()
    base = rec[["rm_id","date"]].drop_duplicates()
    return base.merge(materials_enc_df, on="rm_id", how="left")


def add_days_since_last(df_daily: pd.DataFrame, *, count_today_as_zero: bool = True) -> pd.DataFrame:
    d = df_daily.sort_values(["rm_id", "date"]).copy()
    out = []
    for _, g in d.groupby("rm_id", as_index=False):
        dates = pd.to_datetime(g["date"]).dt.normalize()
        has_deliv = g["y"].to_numpy() > 0
        last_deliv = dates.where(has_deliv).ffill()
        if not count_today_as_zero:
            last_deliv = last_deliv.shift(1)
        ds = (dates - last_deliv).dt.days.astype("float")
        ds = ds.fillna(method="bfill").fillna(len(g))
        out.append(ds)
    d["days_since_last"] = pd.concat(out).sort_index()
    return d


def _compact_reindex_used_values(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        used = sorted(v for v in df[c].unique() if v != 0)
        mapping = {v: i + 1 for i, v in enumerate(used)}
        df[c] = df[c].map(mapping).fillna(0).astype(int)
    return df

def compute_lateness_per_receival(receivals_df: pd.DataFrame, po_df: pd.DataFrame) -> pd.DataFrame:
    recv, po = receivals_df.copy(), po_df.copy()
    recv.columns = recv.columns.str.strip()
    po.columns   = po.columns.str.strip()

    need_recv = {"purchase_order_id", "purchase_order_item_no", "rm_id", "date_arrival"}
    need_po   = {"purchase_order_id", "purchase_order_item_no", "delivery_date"}
    missing_r = need_recv - set(recv.columns)
    missing_p = need_po   - set(po.columns)
    if missing_r:
        raise KeyError(f"Receivals mangler kolonnene: {', '.join(sorted(missing_r))}")
    if missing_p:
        raise KeyError(f"PO-data mangler kolonnene: {', '.join(sorted(missing_p))}")

    recv["date_arrival"] = pd.to_datetime(recv["date_arrival"], errors="coerce").dt.normalize()
    po["delivery_date"]  = pd.to_datetime(po["delivery_date"],  errors="coerce").dt.normalize()
    recv["net_weight"]   = pd.to_numeric(recv.get("net_weight", 0.0), errors="coerce").fillna(0.0)

    agg_recv = (
        recv.groupby(
            ["purchase_order_id", "purchase_order_item_no", "rm_id", "date_arrival"],
            as_index=False
        ).agg(net_weight_sum=("net_weight", "sum"))
    )

    po_unique = (
        po[["purchase_order_id", "purchase_order_item_no", "delivery_date"]]
        .drop_duplicates(subset=["purchase_order_id", "purchase_order_item_no"])
    )

    merged = agg_recv.merge(
        po_unique,
        on=["purchase_order_id", "purchase_order_item_no"],
        how="left",
        suffixes=("", "_po")
    )

    merged["lateness_days"] = (merged["date_arrival"] - merged["delivery_date"]).dt.days

    out = (
        merged[["rm_id", "date_arrival", "delivery_date", "lateness_days", "net_weight_sum"]]
        .sort_values(["rm_id", "date_arrival"])
        .reset_index(drop=True)
    )
    return out

def build_rm_lateness_features(lateness_events: pd.DataFrame) -> pd.DataFrame:
    if lateness_events.empty:
        return pd.DataFrame(columns=["rm_id", "date", "rm_lateness_days", "rm_late_flag"])
    ev = lateness_events.copy()
    ev["date"] = pd.to_datetime(ev["date_arrival"]).dt.normalize()
    ev["lateness_days"] = pd.to_numeric(ev["lateness_days"], errors="coerce").fillna(0.0)
    daily = (
        ev.groupby(["rm_id", "date"], as_index=False)
          .agg(
              rm_lateness_days=("lateness_days", "mean"),
              rm_late_flag=("lateness_days", lambda x: int((x != 0).any()))
          )
          .sort_values(["rm_id", "date"])
          .reset_index(drop=True)
    )
    daily["rm_lateness_days"] = daily["rm_lateness_days"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    daily["rm_late_flag"] = daily["rm_late_flag"].astype(int)
    return daily[["rm_id", "date", "rm_lateness_days", "rm_late_flag"]]


## Pipeline – load & basic cleaning

In [14]:
receivals_df = load_receivals()
po_df        = load_purchase_orders()
materials_df = load_materials_latest()

receivals_df["date_arrival"] = pd.to_datetime(receivals_df["date_arrival"], errors="coerce").dt.normalize()
receivals_df["rm_id"]        = pd.to_numeric(receivals_df["rm_id"], errors="coerce")
receivals_df["net_weight"]   = pd.to_numeric(receivals_df.get("net_weight", 0.0), errors="coerce").fillna(0.0)
receivals_df = receivals_df[
    receivals_df["rm_id"].notna() &
    receivals_df["date_arrival"].notna() &
    (receivals_df["net_weight"] > 0.0)
]

# --- Cutoff ---
if RECEIVALS_MIN_DATE is not None:
    receivals_df = receivals_df[receivals_df["date_arrival"] >= RECEIVALS_MIN_DATE]
if RECEIVALS_MAX_CUTOFF is not None:
    receivals_df = receivals_df[receivals_df["date_arrival"] <= RECEIVALS_MAX_CUTOFF]

# --- Filtrer til valgte rm_id ---
receivals_df = receivals_df[receivals_df["rm_id"].astype(int).isin(INCLUDE_RM_IDS)]


if receivals_df.empty:
    raise RuntimeError("Ingen data for de spesifiserte rm_id etter cutoff.")

# --- Global start ---
first_date_per_rm = receivals_df.groupby("rm_id")["date_arrival"].min()
global_start = first_date_per_rm.min().normalize()

# Sørg for at global_start ikke er eldre enn cutoff
if RECEIVALS_MIN_DATE is not None and global_start < RECEIVALS_MIN_DATE:
    global_start = RECEIVALS_MIN_DATE


# --- Daglig panel ---
date_idx = pd.date_range(global_start, TRAIN_END.normalize(), freq="D")
full_idx = pd.MultiIndex.from_product([sorted(INCLUDE_RM_IDS), date_idx], names=["rm_id","date"])
daily_sum = receivals_df.groupby(["rm_id","date_arrival"], as_index=False)["net_weight"].sum().rename(columns={"date_arrival":"date","net_weight":"y"})
daily_calendar_df = daily_sum.set_index(["rm_id","date"]).reindex(full_idx, fill_value=0.0).reset_index()
daily_calendar_df["y"] = daily_calendar_df["y"].astype(float)

## 3 - Feature engineering

### 3.1 Create features

In [15]:
try:
    lateness_events_df = compute_lateness_per_receival(receivals_df, po_df) if not po_df.empty else pd.DataFrame()
except Exception:
    lateness_events_df = pd.DataFrame()

rm_lateness_df = build_rm_lateness_features(lateness_events_df) if not lateness_events_df.empty else \
                 pd.DataFrame(columns=["rm_id","date","rm_lateness_days","rm_late_flag"])

materials_enc_df = encode_three_material_columns_compact(materials_df)
daily_materials_df = build_daily_material_signals_fast(receivals_df, materials_enc_df)

feature_df = (
    daily_calendar_df
    .merge(rm_lateness_df, on=["rm_id","date"], how="left")
    .merge(daily_materials_df, on=["rm_id","date"], how="left")
    .fillna({"rm_lateness_days":0.0,"rm_late_flag":0,"alloy_code":0,"format_type_code":0,"location_code":0})
)
feature_df = add_days_since_last(feature_df)
feature_df = _compact_reindex_used_values(feature_df, ["alloy_code","format_type_code","location_code"])

# --- rm_id-encoding via pandas category
feature_df["rm_id"] = feature_df["rm_id"].astype(int)
feature_df["rm_id_enc"] = feature_df["rm_id"].astype("category").cat.codes + 1

### 3.2 Dataset building

In [16]:
# --- Build series
past_covariate_cols = ["days_since_last","rm_lateness_days","rm_late_flag","alloy_code","format_type_code","location_code"]
target_series, past_covs_train, past_covs_full, ts_y_full_map = {}, {}, {}, {}

for rm, grp in feature_df.groupby("rm_id"):
    grp = grp.sort_values("date")
    past_cov_df = grp[["date"] + past_covariate_cols]
    ts_y_full = TimeSeries.from_dataframe(grp, time_col="date", value_cols="y", freq="D")
    ts_past_full = TimeSeries.from_dataframe(past_cov_df, time_col="date", value_cols=past_covariate_cols, freq="D")
    rm_code = int(grp["rm_id_enc"].iloc[0])
    static_covs = pd.DataFrame({"rm_id_enc":[rm_code]})
    ts_y_full = ts_y_full.with_static_covariates(static_covs)
    ts_past_full = ts_past_full.with_static_covariates(static_covs)
    ts_y_train = ts_y_full.drop_after(TRAIN_END)
    ts_past_tr = ts_past_full.drop_after(TRAIN_END)
    target_series[int(rm)] = ts_y_train
    past_covs_train[int(rm)] = ts_past_tr
    past_covs_full[int(rm)] = ts_past_full
    ts_y_full_map[int(rm)] = ts_y_full



## 4 - Model 

### 4.1 Traning 

In [17]:
ts_train_list   = [target_series[k]   for k in sorted(target_series.keys())]
past_train_list = [past_covs_train[k] for k in sorted(target_series.keys())]

In [ ]:
BASE_XGB = dict(
    max_depth=8,
    reg_alpha=20,
    reg_lambda=25,
    random_state=42,
    num_leaves=100, 
    verbosity=0,        
)

def make_model() -> XGBModel:
    return XGBModel(
        lags=151,
        lags_past_covariates=151,
        categorical_static_covariates=["rm_id_enc"],  
        output_chunk_length=151,
        **BASE_XGB,
    )


ts_train_list   = [target_series[k]   for k in sorted(target_series.keys())]
past_train_list = [past_covs_train[k] for k in sorted(target_series.keys())]


model = make_model()
model.fit(series=ts_train_list, past_covariates=past_train_list)

XGBModel(lags=151, lags_past_covariates=151, lags_future_covariates=None, output_chunk_length=151, output_chunk_shift=0, add_encoders=None, likelihood=None, quantiles=None, random_state=42, multi_models=True, use_static_covariates=True, categorical_static_covariates=['rm_id_enc'], max_depth=8, reg_alpha=20, reg_lambda=25, num_leaves=100, verbosity=0)

### 4.2 Predictions

In [19]:
horizon_days = (FORECAST_END - FORECAST_START).days + 1
per_rm_predictions = []

for rm, ts_y in target_series.items():
    past_full_ts = past_covs_full[rm]

    ts_y = ts_y.astype(np.float32)
    past_full_ts = past_full_ts.astype(np.float32)

    pred_ts = model.predict(n=horizon_days, series=ts_y, past_covariates=past_full_ts)
    preds = pred_ts.values().flatten().astype(float)

    forecast_index = pd.date_range(FORECAST_START, FORECAST_END, freq="D")
    rm_frame = pd.DataFrame({"rm_id": int(rm), "date": forecast_index, "y_pred_model": preds})

    y_pred = np.maximum(rm_frame["y_pred_model"].to_numpy().astype(float), 0.0)

    df_2024 = feature_df[
        (feature_df["rm_id"] == rm) &
        (feature_df["date"] >= pd.Timestamp("2024-01-01")) &
        (feature_df["date"] <= pd.Timestamp("2024-12-31"))
    ]
    cap_2024 = (1.1 * float(df_2024["y"].max())) if (not df_2024.empty and df_2024["y"].notna().any()) else np.inf
    y_pred = np.minimum(y_pred, cap_2024)

    jan_mask = (forecast_index.month == 1) & (forecast_index.day.isin([1, 2]))
    y_pred[jan_mask] = 0.0

    rm_frame["y_pred"] = y_pred
    per_rm_predictions.append(rm_frame[["rm_id", "date", "y_pred"]])

if not per_rm_predictions:
    raise RuntimeError("Ingen prediksjoner generert.")

pred_daily = pd.concat(per_rm_predictions, ignore_index=True).sort_values(["rm_id", "date"])

pred_daily.to_csv("predictions_daily.csv", index=False)


## 5 - Submission

In [20]:
pred_map = pd.read_csv(PREDICTION_MAPPING_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

pred_daily["date"] = pd.to_datetime(pred_daily["date"])
pred_map["forecast_end_date"] = pd.to_datetime(pred_map["forecast_end_date"])

pred_daily["rm_id"] = pred_daily["rm_id"].round().astype("Int64")
pred_map["rm_id"] = pred_map["rm_id"].round().astype("Int64")

daily_summed = (
    pred_daily.groupby(["rm_id", "date"], as_index=False)["y_pred"]
    .sum()
    .sort_values(["rm_id", "date"])
)
daily_summed["cum_pred"] = daily_summed.groupby("rm_id")["y_pred"].cumsum()

pred_map_unique = pred_map.sort_values(["ID"]).drop_duplicates(subset=["ID"], keep="first")

results = []
for rm, grp_end in pred_map_unique.groupby("rm_id"):
    end_df = grp_end[["ID", "forecast_end_date"]].sort_values("forecast_end_date").reset_index(drop=True)
    hist = daily_summed[daily_summed["rm_id"] == rm][["date", "cum_pred"]].sort_values("date").reset_index(drop=True)
    if hist.empty:
        tmp = end_df.copy()
        tmp["predicted_weight"] = 0.0
    else:
        tmp = pd.merge_asof(
            end_df, hist,
            left_on="forecast_end_date",
            right_on="date",
            direction="backward",
            allow_exact_matches=True
        ).drop(columns=["date"])
        tmp = tmp.rename(columns={"cum_pred": "predicted_weight"})
        tmp["predicted_weight"] = tmp["predicted_weight"].fillna(0.0)
    results.append(tmp[["ID", "predicted_weight"]])

merged_all = pd.concat(results, ignore_index=True)

submission = sample_sub[["ID"]].merge(merged_all, on="ID", how="left")
submission["predicted_weight"] = submission["predicted_weight"].fillna(0.0)

assert not submission["ID"].duplicated().any(), "Duplicate ID values found in submission"

out_path = "best_submission_XGBoost.csv"
submission.to_csv(out_path, index=False)

{"Status": "ok", "rows": len(submission), "file": out_path}

{'Status': 'ok', 'rows': 30450, 'file': 'best_submission_XGBoost.csv'}